# 🚨 Accident Detection from CCTV Footage — CNN Project
**Dataset:** Accident Detection from CCTV Footage (Kaggle - ckay16)
**Models:** CNN from Scratch + Transfer Learning (MobileNetV2) + Data Augmentation
**Author:** Deep Learning Project
---

## 📦 Step 0 — Install & Setup

In [ ]:
import os, warnings, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Flatten, Dense,
                                     Dropout, BatchNormalization, GlobalAveragePooling2D)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

np.random.seed(42)
tf.random.set_seed(42)

os.makedirs('models', exist_ok=True)
os.makedirs('plots', exist_ok=True)

print(f'TensorFlow: {tf.__version__}')
print('GPU Available:', tf.config.list_physical_devices('GPU'))

## 📥 Step 1 — Download Dataset from Kaggle

In [ ]:
# Place your kaggle.json in the same folder as this notebook
# OR set KAGGLE_USERNAME and KAGGLE_KEY environment variables

# Option 1: Copy kaggle.json (run this if you have kaggle.json)
import shutil
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
if os.path.exists('kaggle.json'):
    shutil.copy('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
    os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
    print('kaggle.json copied!')

# Download dataset
os.system('pip install kaggle -q')
os.system('kaggle datasets download -d ckay16/accident-detection-from-cctv-footage')

# Extract
import zipfile
with zipfile.ZipFile('accident-detection-from-cctv-footage.zip', 'r') as z:
    z.extractall('data')

print('Dataset downloaded and extracted!')
print(os.listdir('data'))

## 🔍 Step 2 — EDA & Visualisation

In [ ]:
# Check dataset structure
DATA_DIR = 'data'
for split in ['train', 'val', 'test']:
    split_path = os.path.join(DATA_DIR, split)
    if os.path.exists(split_path):
        for cls in os.listdir(split_path):
            cls_path = os.path.join(split_path, cls)
            if os.path.isdir(cls_path):
                count = len(os.listdir(cls_path))
                print(f'{split}/{cls}: {count} images')

In [ ]:
# Visualise sample images
from tensorflow.keras.preprocessing.image import load_img

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
classes = ['Accident', 'Non Accident']

for row, cls in enumerate(classes):
    cls_dir = os.path.join(DATA_DIR, 'train', cls)
    imgs = os.listdir(cls_dir)[:5]
    for col, img_name in enumerate(imgs):
        img = load_img(os.path.join(cls_dir, img_name), target_size=(224, 224))
        axes[row][col].imshow(img)
        axes[row][col].set_title(cls, fontweight='bold',
                                  color='red' if cls=='Accident' else 'green')
        axes[row][col].axis('off')

plt.suptitle('Sample Images from Dataset', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/01_sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Class distribution
counts = {}
for split in ['train', 'val', 'test']:
    split_path = os.path.join(DATA_DIR, split)
    if os.path.exists(split_path):
        for cls in os.listdir(split_path):
            cls_path = os.path.join(split_path, cls)
            if os.path.isdir(cls_path):
                key = f'{split}/{cls}'
                counts[key] = len(os.listdir(cls_path))

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#ef4444' if 'Accident' in k else '#22c55e' for k in counts]
bars = ax.bar(list(counts.keys()), list(counts.values()), color=colors, edgecolor='white', linewidth=1.5)
for bar, v in zip(bars, counts.values()):
    ax.text(bar.get_x()+bar.get_width()/2, v+5, str(v), ha='center', fontweight='bold')
ax.set_title('Class Distribution Across Splits', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Images')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('plots/02_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 🏗️ Step 3 — Data Preprocessing & Augmentation

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Basic generators (for CNN scratch & Transfer Learning)
basic_gen = ImageDataGenerator(rescale=1./255)

train_data = basic_gen.flow_from_directory(
    os.path.join(DATA_DIR, 'train'),
    target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', shuffle=True)

val_data = basic_gen.flow_from_directory(
    os.path.join(DATA_DIR, 'val'),
    target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', shuffle=False)

test_data = basic_gen.flow_from_directory(
    os.path.join(DATA_DIR, 'test'),
    target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', shuffle=False)

# Augmented generators
aug_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1,
    brightness_range=[0.8, 1.2]
)

train_aug = aug_gen.flow_from_directory(
    os.path.join(DATA_DIR, 'train'),
    target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', shuffle=True)

CLASS_NAMES = list(train_data.class_indices.keys())
print('Classes:', CLASS_NAMES)
print('Class indices:', train_data.class_indices)

In [ ]:
# Save class names for app
with open('models/class_names.pkl', 'wb') as f:
    pickle.dump(CLASS_NAMES, f)
print('Class names saved:', CLASS_NAMES)

In [ ]:
# Visualise augmented images
sample_batch = next(train_aug)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(sample_batch[0][i])
    lbl = 'Accident' if sample_batch[1][i] == 1 else 'Non Accident'
    ax.set_title(lbl, color='red' if lbl=='Accident' else 'green', fontweight='bold')
    ax.axis('off')
plt.suptitle('Augmented Training Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/03_augmented_images.png', dpi=150, bbox_inches='tight')
plt.show()

## 🧠 Step 4 — Model 1: CNN from Scratch

In [ ]:
cnn_scratch = Sequential([
    # Block 1
    Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3), padding='same'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    # Block 2
    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    # Block 3
    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    # Block 4
    Conv2D(256, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    # Classifier
    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

cnn_scratch.compile(optimizer=Adam(0.001), loss='binary_crossentropy',
                    metrics=['accuracy'])
cnn_scratch.summary()

In [ ]:
callbacks_scratch = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint('models/cnn_scratch_best.keras', monitor='val_accuracy',
                    save_best_only=True, verbose=0),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
]

history_scratch = cnn_scratch.fit(
    train_data, validation_data=val_data,
    epochs=20, callbacks=callbacks_scratch, verbose=1
)

## 🔁 Step 5 — Model 2: Transfer Learning (MobileNetV2)

In [ ]:
base_model = MobileNetV2(input_shape=(224,224,3), include_top=False, weights='imagenet')
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(1, activation='sigmoid')(x)

model_tl = Model(inputs=base_model.input, outputs=output)
model_tl.compile(optimizer=Adam(0.001), loss='binary_crossentropy', metrics=['accuracy'])

print(f'Total params: {model_tl.count_params():,}')
print(f'Trainable params: {sum([tf.size(w).numpy() for w in model_tl.trainable_weights]):,}')

In [ ]:
callbacks_tl = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint('models/transfer_learning_best.keras', monitor='val_accuracy',
                    save_best_only=True, verbose=0),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
]

history_tl = model_tl.fit(
    train_data, validation_data=val_data,
    epochs=15, callbacks=callbacks_tl, verbose=1
)

## 🔓 Step 6 — Fine Tuning Transfer Learning Model

In [ ]:
# Unfreeze last 20 layers
for layer in model_tl.layers[-20:]:
    layer.trainable = True

model_tl.compile(optimizer=Adam(1e-5), loss='binary_crossentropy', metrics=['accuracy'])

callbacks_ft = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint('models/fine_tuned_best.keras', monitor='val_accuracy',
                    save_best_only=True, verbose=0),
]

history_ft = model_tl.fit(
    train_aug, validation_data=val_data,
    epochs=10, callbacks=callbacks_ft, verbose=1
)

## 📊 Step 7 — Evaluation & Comparison

In [ ]:
def evaluate_model(model, test_data, name):
    test_data.reset()
    loss, acc = model.evaluate(test_data, verbose=0)
    test_data.reset()
    y_prob = model.predict(test_data, verbose=0).ravel()
    y_true = test_data.classes
    y_pred = (y_prob >= 0.5).astype(int)
    sep = '='*45
    print(f'\n{sep}')
    print(f'  {name}')
    print(sep)
    print(f'  Accuracy : {acc*100:.2f}%')
    print(f'  Loss     : {loss:.4f}')
    print(f'\n{classification_report(y_true, y_pred, target_names=CLASS_NAMES)}')
    return {'name':name,'acc':acc,'loss':loss,'y_prob':y_prob,'y_pred':y_pred,'y_true':y_true}

r1 = evaluate_model(cnn_scratch, test_data, 'CNN from Scratch')
r2 = evaluate_model(model_tl,    test_data, 'Transfer Learning (MobileNetV2)')

In [ ]:
# Training history plots
def plot_history(history, title, savepath):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history.history['accuracy'], label='Train', color='#3b82f6', lw=2)
    axes[0].plot(history.history['val_accuracy'], label='Val', color='#ef4444', lw=2)
    axes[0].set_title(f'{title} — Accuracy', fontweight='bold')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy'); axes[0].legend()
    axes[1].plot(history.history['loss'], label='Train', color='#3b82f6', lw=2)
    axes[1].plot(history.history['val_loss'], label='Val', color='#ef4444', lw=2)
    axes[1].set_title(f'{title} — Loss', fontweight='bold')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss'); axes[1].legend()
    plt.tight_layout()
    plt.savefig(savepath, dpi=150, bbox_inches='tight')
    plt.show()

plot_history(history_scratch, 'CNN from Scratch',   'plots/04_cnn_scratch_history.png')
plot_history(history_ft,      'Transfer Learning',  'plots/05_transfer_learning_history.png')

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, r in zip(axes, [r1, r2]):
    cm = confusion_matrix(r['y_true'], r['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                linewidths=1, linecolor='white')
    ax.set_title(f"{r['name']}\nAcc={r['acc']*100:.2f}%", fontweight='bold')
plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/06_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Model comparison
results = [r1, r2]
names   = [r['name'] for r in results]
accs    = [r['acc']*100 for r in results]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(names, accs, color=['#3b82f6','#22c55e'], edgecolor='white', linewidth=2, width=0.4)
for bar, v in zip(bars, accs):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.3, f'{v:.2f}%', ha='center', fontweight='bold', fontsize=11)
ax.set_title('Model Comparison — Test Accuracy', fontsize=14, fontweight='bold')
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 110)
plt.tight_layout()
plt.savefig('plots/07_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

best_idx = accs.index(max(accs))
print(f'\n✅ Best Model: {names[best_idx]} with {accs[best_idx]:.2f}% accuracy')

## 💾 Step 8 — Save Best Model

In [ ]:
# Save best model (Transfer Learning is usually best)
model_tl.save('models/accident_detection_model.h5')
print('✅ Best model saved as models/accident_detection_model.h5')

# Also save class names
with open('models/class_names.pkl', 'wb') as f:
    pickle.dump(CLASS_NAMES, f)
print('✅ Class names saved')

print('\n=== PROJECT COMPLETE ===')
print(f'CNN Scratch Accuracy    : {r1["acc"]*100:.2f}%')
print(f'Transfer Learning Acc   : {r2["acc"]*100:.2f}%')
print('Model saved to          : models/accident_detection_model.h5')